# LLM Narrative Generation + Fact Check Notebook

This notebook accepts:
1. **A plot description** (or figure/chart summary)
2. **Source narratives / evidence text**

It then:
- asks an LLM to generate a narrative grounded in the inputs
- asks the LLM to fact-check the generated narrative against the provided evidence
- returns structured JSON output for easier downstream use

## What you need
- Python 3.9+
- An LLM API key in your environment
- A model that can follow JSON instructions reliably

By default, this notebook uses the `openai` Python package and an OpenAI-compatible chat interface.


In [ ]:
# Optional: install dependencies if needed
# !pip install openai pandas python-dotenv


In [ ]:
import os
import json
from textwrap import dedent
from typing import Any, Dict, List

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

from openai import OpenAI


## Configuration

Set your API key in the environment before running, for example:

- macOS/Linux: `export OPENAI_API_KEY='your_key_here'`
- Windows PowerShell: `$env:OPENAI_API_KEY='your_key_here'`

If you are using another OpenAI-compatible endpoint, set `BASE_URL` below.


In [ ]:
MODEL_NAME = os.getenv('LLM_MODEL', 'gpt-4.1-mini')
BASE_URL = os.getenv('OPENAI_BASE_URL')  # optional

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=BASE_URL if BASE_URL else None,
)

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('OPENAI_API_KEY is not set in your environment.')

print('Using model:', MODEL_NAME)
print('Base URL:', BASE_URL or 'default OpenAI endpoint')


## Input data

Replace the example values below with your own plot summary and evidence.

Recommended practice:
- Put only verifiable source facts into `evidence_items`
- Keep `plot_input` descriptive and concise
- Include numeric values and axis trends whenever possible


In [ ]:
plot_input = {
    'title': 'Quarterly revenue by region',
    'chart_type': 'line chart',
    'x_axis': 'Quarter',
    'y_axis': 'Revenue (USD millions)',
    'visual_summary': (
        'North America rises steadily from Q1 to Q4. '
        'Europe dips slightly in Q2 before recovering. '
        'APAC shows the fastest growth in the second half of the year.'
    ),
    'data_points': [
        {'region': 'North America', 'quarter': 'Q1', 'value': 120},
        {'region': 'North America', 'quarter': 'Q2', 'value': 128},
        {'region': 'North America', 'quarter': 'Q3', 'value': 135},
        {'region': 'North America', 'quarter': 'Q4', 'value': 142},
        {'region': 'Europe', 'quarter': 'Q1', 'value': 98},
        {'region': 'Europe', 'quarter': 'Q2', 'value': 94},
        {'region': 'Europe', 'quarter': 'Q3', 'value': 101},
        {'region': 'Europe', 'quarter': 'Q4', 'value': 109},
        {'region': 'APAC', 'quarter': 'Q1', 'value': 70},
        {'region': 'APAC', 'quarter': 'Q2', 'value': 76},
        {'region': 'APAC', 'quarter': 'Q3', 'value': 95},
        {'region': 'APAC', 'quarter': 'Q4', 'value': 118},
    ]
}

evidence_items = [
    'North America revenue increased every quarter from Q1 to Q4.',
    'Europe decreased from 98 in Q1 to 94 in Q2, then increased to 109 by Q4.',
    'APAC grew from 70 in Q1 to 118 in Q4, the largest absolute gain among the regions.',
    'All values are reported in USD millions.'
]

generation_style = {
    'audience': 'general business audience',
    'tone': 'clear, analytical, concise',
    'length': '1 short paragraph',
    'must_avoid': [
        'unsupported causal claims',
        'invented numbers',
        'claims not grounded in evidence'
    ]
}

input_payload = {
    'plot_input': plot_input,
    'evidence_items': evidence_items,
    'generation_style': generation_style,
}

print(json.dumps(input_payload, indent=2, ensure_ascii=False))


## Helper: call the LLM

This helper asks for JSON output only.


In [ ]:
def llm_json(system_prompt: str, user_prompt: str, model: str = MODEL_NAME) -> Dict[str, Any]:
    response = client.chat.completions.create(
        model=model,
        temperature=0.2,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )

    content = response.choices[0].message.content
    return json.loads(content)


## Step 1: Generate a grounded narrative

The model is instructed to use only the supplied evidence.


In [ ]:
generation_system_prompt = dedent('''
You are a careful data storytelling assistant.
Generate a narrative ONLY from the supplied plot description and evidence.
Do not invent facts, causes, explanations, or numbers.
If something is uncertain, say so conservatively.
Return valid JSON only.
''').strip()

generation_user_prompt = dedent(f'''
Create a grounded narrative from the input below.

Return JSON with this schema:
{{
  "narrative": "string",
  "claims": [
    {{"claim": "string", "evidence_refs": [0, 1]}}
  ],
  "uncertainties": ["string"]
}}

Input:
{json.dumps(input_payload, indent=2, ensure_ascii=False)}
''').strip()

generated = llm_json(generation_system_prompt, generation_user_prompt)
generated


## Step 2: Fact-check the generated narrative

This pass verifies each claim against the supplied evidence only.


In [ ]:
factcheck_system_prompt = dedent('''
You are a strict fact-checker.
Check the generated narrative ONLY against the provided evidence.
Do not use outside knowledge.
Return valid JSON only.
''').strip()

factcheck_input = {
    'plot_input': plot_input,
    'evidence_items': evidence_items,
    'generated_output': generated,
}

factcheck_user_prompt = dedent(f'''
Evaluate whether the generated narrative is supported by the provided evidence.

Return JSON with this schema:
{{
  "overall_verdict": "supported | partially_supported | unsupported",
  "claim_checks": [
    {{
      "claim": "string",
      "status": "supported | partially_supported | unsupported",
      "reason": "string",
      "supporting_evidence_refs": [0, 1],
      "suggested_revision": "string"
    }}
  ],
  "revised_narrative": "string"
}}

Input:
{json.dumps(factcheck_input, indent=2, ensure_ascii=False)}
''').strip()

factcheck = llm_json(factcheck_system_prompt, factcheck_user_prompt)
factcheck


## Step 3: Pretty-print results


In [ ]:
print('GENERATED NARRATIVE')
print('-' * 80)
print(generated.get('narrative', ''))
print()

print('FACT CHECK VERDICT')
print('-' * 80)
print(factcheck.get('overall_verdict', ''))
print()

print('CLAIM CHECKS')
print('-' * 80)
for i, item in enumerate(factcheck.get('claim_checks', []), start=1):
    print(f"{i}. Claim: {item.get('claim', '')}")
    print(f"   Status: {item.get('status', '')}")
    print(f"   Reason: {item.get('reason', '')}")
    print(f"   Evidence refs: {item.get('supporting_evidence_refs', [])}")
    print(f"   Suggested revision: {item.get('suggested_revision', '')}")
    print()

print('REVISED NARRATIVE')
print('-' * 80)
print(factcheck.get('revised_narrative', ''))


## Optional: wrap into a reusable function


In [ ]:
def generate_and_factcheck(
    plot_input: Dict[str, Any],
    evidence_items: List[str],
    generation_style: Dict[str, Any],
    model: str = MODEL_NAME,
) -> Dict[str, Any]:
    input_payload = {
        'plot_input': plot_input,
        'evidence_items': evidence_items,
        'generation_style': generation_style,
    }

    generation_user_prompt = dedent(f'''
    Create a grounded narrative from the input below.

    Return JSON with this schema:
    {{
      "narrative": "string",
      "claims": [
        {{"claim": "string", "evidence_refs": [0, 1]}}
      ],
      "uncertainties": ["string"]
    }}

    Input:
    {json.dumps(input_payload, indent=2, ensure_ascii=False)}
    ''').strip()

    generated = llm_json(generation_system_prompt, generation_user_prompt, model=model)

    factcheck_input = {
        'plot_input': plot_input,
        'evidence_items': evidence_items,
        'generated_output': generated,
    }

    factcheck_user_prompt = dedent(f'''
    Evaluate whether the generated narrative is supported by the provided evidence.

    Return JSON with this schema:
    {{
      "overall_verdict": "supported | partially_supported | unsupported",
      "claim_checks": [
        {{
          "claim": "string",
          "status": "supported | partially_supported | unsupported",
          "reason": "string",
          "supporting_evidence_refs": [0, 1],
          "suggested_revision": "string"
        }}
      ],
      "revised_narrative": "string"
    }}

    Input:
    {json.dumps(factcheck_input, indent=2, ensure_ascii=False)}
    ''').strip()

    factcheck = llm_json(factcheck_system_prompt, factcheck_user_prompt, model=model)

    return {
        'generated': generated,
        'factcheck': factcheck,
    }


## Example function call


In [ ]:
result = generate_and_factcheck(
    plot_input=plot_input,
    evidence_items=evidence_items,
    generation_style=generation_style,
)

print(json.dumps(result, indent=2, ensure_ascii=False))


## Notes

- This notebook **checks claims only against the evidence you provide**.
- For stronger fact checking, give it structured evidence such as extracted chart values, annotations, source captions, or trusted text snippets.
- If you want citation-style spans, you can extend the prompts so each claim points to exact evidence items or document chunks.
- If your plots come from images, you can add a preprocessing step that extracts chart values before calling the LLM.
